# 04 – Train Autoencoder

Trains an MLP bottleneck autoencoder for reconstruction-error-based anomaly detection. Saves artefacts to `models/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings("ignore")


In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
CONTAMINATION = 0.08

FEAT_FILE = ROOT / "data" / "processed" / "features.parquet"
feat_df = pd.read_parquet(str(FEAT_FILE))
FEAT_COLS = [
    "requests_per_hour", "error_rate", "unique_endpoints",
    "avg_bytes_sent", "post_ratio", "is_off_hours", "is_weekend", "has_scanner_ua"
]
X = feat_df[FEAT_COLS].to_numpy(dtype=float)
print(f"Training on {X.shape[0]:,} samples × {X.shape[1]} features")


In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
ae_scaler = StandardScaler()
X_scaled = ae_scaler.fit_transform(X)

MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(ae_scaler, str(MODELS_DIR / "autoencoder_scaler.joblib"))


In [ ]:
# ── Architecture: input → hidden → bottleneck → hidden → reconstruction ───────
n_features = X_scaled.shape[1]
bottleneck  = max(2, n_features // 2)
hidden      = max(4, n_features * 2)

ae = MLPRegressor(
    hidden_layer_sizes=(hidden, bottleneck, hidden),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=RANDOM_STATE,
    verbose=False,
)
ae.fit(X_scaled, X_scaled)
print(f"Autoencoder trained: {hidden}→{bottleneck}→{hidden} neurons")
print(f"  Loss (final iteration): {ae.loss_:.6f}")


In [ ]:
# ── Reconstruction error ──────────────────────────────────────────────────────
X_recon = ae.predict(X_scaled)
recon_error = np.mean((X_scaled - X_recon) ** 2, axis=1)

threshold = np.percentile(recon_error, (1 - CONTAMINATION) * 100)
ae_flags = recon_error > threshold
print(f"Reconstruction error: mean={recon_error.mean():.4f}, p95={np.percentile(recon_error,95):.4f}")
print(f"Threshold ({(1-CONTAMINATION)*100:.0f}th pct): {threshold:.4f}")
print(f"Anomalies flagged: {ae_flags.sum()} / {len(ae_flags)}")


In [ ]:
# ── Save autoencoder ──────────────────────────────────────────────────────────
joblib.dump(ae, str(MODELS_DIR / "autoencoder_ae.joblib"))
print("Autoencoder saved → models/autoencoder_ae.joblib")

# ── Plot reconstruction error distribution ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(recon_error, bins=50, color="steelblue", alpha=0.8, label="Reconstruction error")
ax.axvline(threshold, color="red", linestyle="--", label=f"Threshold ({threshold:.4f})")
ax.set_xlabel("Mean Squared Reconstruction Error")
ax.set_ylabel("Count")
ax.set_title("Autoencoder Reconstruction Error Distribution")
ax.legend()
plt.tight_layout(); plt.show()
